# Fluent warm start from a Python script

The same steps as `flowtrace fluent-warmstart`, from Python: read a Fluent mesh, check its
quality, solve, and write the interpolation file plus the scripts that continue *your* case in
Fluent. A small mesh is generated here so the notebook runs anywhere. With a real case, pass the
path of your `.msh`/`.cas.h5` file instead.

In [1]:
import os, tempfile, json
import numpy as np
from flowtrace.mesh.unstructured import cartesian_mesh_2d
from flowtrace.mesh.fluent_msh import write_fluent_mesh
from flowtrace.mesh.quality import mesh_quality

work = tempfile.mkdtemp(prefix="flowtrace_ws_")
mesh = cartesian_mesh_2d(40, 10, 4.0, 1.0, names=("inlet", "outlet", "bottom", "top"),
                         types=("velocity-inlet", "pressure-outlet", "wall", "wall"), triangulate=True, warp=0.02)
case_file = os.path.join(work, "duct.msh")
write_fluent_mesh(mesh, case_file)
print(mesh_quality(mesh).summary())

Mesh Quality:

  Minimum Orthogonal Quality = 6.79346e-01  cell 198 at (0.93551, 0.92554)
  (Fluent Meshing reports the inverse: 1 - orthogonal quality)

  Maximum Equiangle Skewness = 5.02203e-01  cell 588 at (2.9355, 0.42554)
  Maximum Aspect Ratio = 1.85694e+00  cell 568 at (2.8363, 0.42574)
  Non-orthogonality: max 47.18 deg, mean 18.26 deg
  Maximum neighbour volume ratio = 1.079
  Volume: min 4.32609e-03, total 4
  Left-handed faces repaired: 0

  No quality problems found (Fluent recommendations).


In [2]:
from flowtrace.fluent.warmstart import run_warmstart
from flowtrace.unstructured.case import BoundarySpec, UnstructuredCase

case = UnstructuredCase(boundaries={"inlet": BoundarySpec("inlet", normal_speed=1.0)}, viscosity=0.02,
                        max_iter=2000, tol=1e-6, backend="numpy")
report = run_warmstart(case_file, case, os.path.join(work, "ws"), log=lambda *a: None)
print(f"FlowTrace: {report['iterations']} iterations, converged: {report['converged']}, {report['solve_seconds']:.1f} s")
for k, v in report["files"].items():
    print(f"  {k:<7} {os.path.basename(v)}")

FlowTrace: 137 iterations, converged: True, 1.0 s
  ip      flowtrace.ip
  jou     warmstart.jou
  py      warmstart_pyfluent.py
  report  report.json
  readme  README_WARMSTART.txt


In [3]:
print(open(report["files"]["jou"]).read())

; FlowTrace warm start for duct.msh
; Ansys and Fluent are registered trademarks of ANSYS, Inc. FlowTrace is an independent project; it is compatible with, but not affiliated with or endorsed by, ANSYS, Inc.
/file/read-case "C:/Users/Hamid/AppData/Local/Temp/flowtrace_ws_b420mkou/duct.msh"
/solve/initialize/initialize-flow

/file/interpolate/read-data "C:/Users/Hamid/AppData/Local/Temp/flowtrace_ws_b420mkou/ws/flowtrace.ip"


/solve/iterate 500
/file/write-case-data "warmstart_result.cas.h5"
/exit yes



## How much does the warm start save?

Fluent itself needs a licence, so the measurement below uses FlowTrace's own SIMPLEC solver as a
proxy. The `.ip` file is interpolated onto a *different* mesh. The cold run is converged tightly
and its final field is the reference answer; both runs are counted until the velocity field is
within 10⁻⁴ (relative, max norm) of that answer.

In [4]:
from flowtrace.fluent import warmstart_study as W
W.CASES["tutorial"] = {"title": "cavity Re=100", "preset": "cavity", "params": {"re": 100, "n": 48},
                       "target": {"kind": "tri", "n": 32}}
r = W.run_proxy_case("tutorial", os.path.join(work, "study"), backend="numpy")
print(f"to the converged field: cold start {r.cold_iterations} iterations, warm start {r.warm_iterations} "
      f"iterations (x{r.speedup:.1f}); warm field within {r.max_velocity_difference:.1e} of the cold solution")

E:\Code archives\gputest\.venv\Lib\site-packages\cupy\_environment.py:286: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


to the converged field: cold start 872 iterations, warm start 306 iterations (x2.8); warm field within 1.0e-04 of the cold solution
